In [25]:
import duckdb
con = duckdb.connect('../data/practice.duckdb')
con.sql("SET memory_limit = '4GB'")
con.sql("SET preserve_insertion_order = false")
con.sql("SET temp_directory = '../data/tmp'")

# 07 · Stockout flag (censored demand)
**Business question:** which zero-sales days are real zero demand, and which are likely stockouts where customers wanted the item but the shelf was empty?

**Why it matters:** treating stockout zeros as real demand pulls the forecast down, inflates σ and safety stock, and teaches an ML model that demand collapses for no reason.


** Q: The planning lead wants to see what happened to FOODS_3_090 in store CA_3 between 2011-02-01 and 2011-03-10. Show each day’s date, units and sell price, plus a column zero_day that shows 1 when nothing sold and 0 otherwise. Oldest day first.

Checking sales daily table before writing actual query

In [4]:
con.sql("""SELECT * FROM sales_daily WHERE store_id = 'CA_3' AND item_id = 'FOODS_3_090' LIMIT 5""").df()

,item_id,dept_id,cat_id,store_id,state_id,date,wm_yr_wk,units,sell_price,revenue
0,FOODS_3_090,FOODS_3,FOODS,CA_3,CA,2011-01-29,11101,108,1.25,135.0
1,FOODS_3_090,FOODS_3,FOODS,CA_3,CA,2011-01-30,11101,132,1.25,165.0
2,FOODS_3_090,FOODS_3,FOODS,CA_3,CA,2011-01-31,11101,102,1.25,127.5
3,FOODS_3_090,FOODS_3,FOODS,CA_3,CA,2011-02-01,11101,120,1.25,150.0
4,FOODS_3_090,FOODS_3,FOODS,CA_3,CA,2011-02-02,11101,106,1.25,132.5


** Query for the problem

In [10]:
con.sql("""SELECT
            store_id,
            item_id,
            date,
            units,
            sell_price,
            CASE WHEN units = 0 THEN 1 ELSE 0 END AS zero_day
            FROM sales_daily
            WHERE store_id = 'CA_3'
            AND item_id = 'FOODS_3_090'
            AND date BETWEEN '2011-02-01' AND '2011-03-10'
            ORDER BY date""").df()

,store_id,item_id,date,units,sell_price,zero_day
0,CA_3,FOODS_3_090,2011-02-01,120,1.25,0
1,CA_3,FOODS_3_090,2011-02-02,106,1.25,0
2,CA_3,FOODS_3_090,2011-02-03,123,1.25,0
3,CA_3,FOODS_3_090,2011-02-04,279,1.25,0
4,CA_3,FOODS_3_090,2011-02-05,175,1.25,0
5,CA_3,FOODS_3_090,2011-02-06,186,1.25,0
6,CA_3,FOODS_3_090,2011-02-07,120,1.25,0
7,CA_3,FOODS_3_090,2011-02-08,0,1.25,1
8,CA_3,FOODS_3_090,2011-02-09,0,1.25,1
9,CA_3,FOODS_3_090,2011-02-10,0,1.25,1


** Q. Save a table item_store_rate with one row per item × store showing: total on-sale days, average daily units, and the % of on-sale days with zero sales. Only count days when the item was on sale.

In [16]:
con.sql("""CREATE OR REPLACE TABLE item_store_rate AS
SELECT store_id,
       item_id,
       COUNT(*)                                           AS on_sale_days,
       AVG(units)                                         AS avg_daily_units,
       SUM(CASE WHEN units = 0 THEN 1 ELSE 0 END)         AS zero_sale_days,
       SUM(CASE WHEN units = 0 THEN 1 ELSE 0 END) * 100.0
         / COUNT(*)                                       AS pct_zero_days
FROM sales_daily
WHERE sell_price IS NOT NULL          -- on-sale days only, for every column
GROUP BY store_id, item_id""")

In [17]:
con.sql("""SELECT * FROM item_store_rate ORDER BY avg_daily_units DESC LIMIT 10""").df()

,store_id,item_id,on_sale_days,avg_daily_units,zero_sale_days,pct_zero_days
0,CA_3,FOODS_3_090,1941,130.787738,359.0,18.495621
1,TX_2,FOODS_3_586,1941,100.525502,8.0,0.412159
2,TX_3,FOODS_3_586,1941,78.239052,7.0,0.360639
3,CA_3,FOODS_3_586,1941,70.205564,5.0,0.257599
4,CA_1,FOODS_3_090,1941,66.385884,469.0,24.162803
5,WI_3,FOODS_3_090,1941,63.626996,487.0,25.090160
6,TX_2,FOODS_3_090,1941,62.480680,471.0,24.265842
7,TX_3,FOODS_3_090,1941,60.161257,472.0,24.317362
8,TX_2,FOODS_3_252,1941,59.563627,4.0,0.206079
9,TX_1,FOODS_3_586,1941,58.737764,14.0,0.721278


In [21]:
con.sql("""SELECT 
            store_id,
            item_id,
            units,
            date,
            CASE WHEN units>0 THEN 0 ELSE 1 END AS zero_sale_day
            FROM sales_daily 
            WHERE sell_price IS NOT NULL
            AND units = 0
            AND store_id = 'CA_3'
            AND item_id = 'FOODS_3_090'""").df()
            

,store_id,item_id,units,date,zero_sale_day
0,CA_3,FOODS_3_090,0,2011-02-08,1
1,CA_3,FOODS_3_090,0,2011-02-09,1
2,CA_3,FOODS_3_090,0,2011-02-10,1
3,CA_3,FOODS_3_090,0,2011-02-11,1
4,CA_3,FOODS_3_090,0,2011-02-12,1
...,...,...,...,...,...
354,CA_3,FOODS_3_090,0,2016-01-02,1
355,CA_3,FOODS_3_090,0,2016-01-03,1
356,CA_3,FOODS_3_090,0,2016-01-04,1
357,CA_3,FOODS_3_090,0,2016-01-05,1


In [24]:
    con.sql("""WITH zero_day AS (
                SELECT 
                store_id,
                item_id,
                date,
                date - CAST( ROW_NUMBER() OVER (PARTITION BY store_id, item_id ORDER BY date) AS INTEGER) AS grp
                FROM sales_daily
                WHERE store_id = 'CA_3'
                AND item_id = 'FOODS_3_090'
                AND sell_price IS NOT NULL
                AND units = 0 )
                SELECT 
                store_id,
                item_id,
                grp,
                MIN(date) AS run_start_date,
                MAX(date) AS run_end_date,
                COUNT(*) AS run_length
                FROM zero_day
                GROUP BY store_id, item_id, grp
                ORDER BY run_length DESC""").df()
                
            

,store_id,item_id,grp,run_start_date,run_end_date,run_length
0,CA_3,FOODS_3_090,2011-02-07,2011-02-08,2011-09-24,229
1,CA_3,FOODS_3_090,2015-01-12,2015-12-29,2016-01-06,9
2,CA_3,FOODS_3_090,2014-04-22,2015-03-06,2015-03-11,6
3,CA_3,FOODS_3_090,2014-03-02,2014-12-31,2015-01-04,5
4,CA_3,FOODS_3_090,2014-02-17,2014-12-01,2014-12-05,5
...,...,...,...,...,...,...
58,CA_3,FOODS_3_090,2013-05-29,2014-02-24,2014-02-24,1
59,CA_3,FOODS_3_090,2013-08-27,2014-06-04,2014-06-04,1
60,CA_3,FOODS_3_090,2014-04-15,2015-02-25,2015-02-25,1
61,CA_3,FOODS_3_090,2014-07-03,2015-05-31,2015-05-31,1


In [28]:
    con.sql("""CREATE OR REPLACE TABLE zero_runs AS
                WITH zero_day AS (
                SELECT 
                store_id,
                item_id,
                date,
                date - CAST( ROW_NUMBER() OVER (PARTITION BY store_id, item_id ORDER BY date) AS INTEGER) AS grp
                FROM sales_daily
                WHERE sell_price IS NOT NULL
                AND units = 0 )
                SELECT 
                store_id,
                item_id,
                MIN(date) AS run_start_date,
                MAX(date) AS run_end_date,
                COUNT(*) AS run_length
                FROM zero_day
                GROUP BY store_id, item_id, grp
                ORDER BY run_length DESC""")
                
            

**Q. Save a table stockout_runs with one row per zero-sales run, for every item in every store (on-sale days only), showing: store, item, run start, run end, run length, the item-store’s zero rate as a fraction, the chance of a run that long, and a column stockout_flag = 1 when the chance is below 0.001 and the run is at least 3 days, otherwise 0.

In [29]:
con.sql("""SELECT * FROM zero_runs LIMIT 10""").df()
            

,store_id,item_id,run_start_date,run_end_date,run_length
0,TX_2,HOUSEHOLD_1_199,2012-02-25,2012-03-18,23
1,CA_1,FOODS_2_129,2016-04-22,2016-05-14,23
2,TX_1,HOUSEHOLD_2_079,2014-06-03,2014-06-25,23
3,CA_3,FOODS_3_762,2013-05-14,2013-06-05,23
4,TX_1,HOUSEHOLD_1_433,2015-07-31,2015-08-22,23
5,WI_1,FOODS_3_346,2011-03-30,2011-04-21,23
6,WI_3,FOODS_3_648,2016-01-12,2016-02-03,23
7,TX_1,HOBBIES_1_149,2015-12-23,2016-01-14,23
8,TX_3,HOUSEHOLD_2_398,2013-05-17,2013-06-08,23
9,TX_2,HOBBIES_1_280,2016-04-28,2016-05-20,23


In [31]:
con.sql("""SELECT (SELECT SUM(run_length)     FROM zero_runs)       AS days_in_runs,
       (SELECT SUM(zero_sale_days) FROM item_store_rate) AS zero_days_step2,
       (SELECT COUNT(*)            FROM zero_runs)       AS total_runs,
       (SELECT MAX(run_length)     FROM zero_runs)       AS longest_run""").df()

,days_in_runs,zero_days_step2,total_runs,longest_run
0,27942406.0,27942406.0,6613508,1770


**4b. Using zero_runs and item_store_rate, save a table stockout_runs with everything from zero_runs plus:

the item-store’s zero rate as a fraction (p)
the chance of a run that long (p to the power of the run length)
stockout_flag = 1 when the chance is below 0.001 and the run is at least 3 days, otherwise 0

In [45]:
con.sql("""CREATE OR REPLACE TABLE stockout_runs AS 
            SELECT
            z.store_id,
            z.item_id,
            z.run_start_date,
            z.run_end_date,
            z.run_length,
            i.pct_zero_days/100.0 AS zero_rate_fraction_p,
            power(i.pct_zero_days/100.0, z.run_length) AS chance,
            CASE WHEN power(i.pct_zero_days/100.0, z.run_length) < 0.001 AND z.run_length>=3 THEN 1 ELSE 0 END AS stockout_flag
            FROM zero_runs z
            LEFT JOIN item_store_rate i
                ON z.store_id = i.store_id
                AND z.item_id = i.item_id""")

In [47]:
con.sql("""SELECT *
FROM stockout_runs
WHERE stockout_flag = 1
ORDER BY run_length DESC, run_start_date
LIMIT 2""").df()

,store_id,item_id,run_start_date,run_end_date,run_length,zero_rate_fraction_p,chance,stockout_flag
0,CA_4,HOUSEHOLD_1_512,2011-04-18,2016-02-20,1770,0.989181,4.344874e-09,1
1,TX_3,FOODS_2_107,2011-02-21,2015-11-09,1723,0.973725,1.190395e-20,1


In [53]:
con.sql("""CREATE OR REPLACE TABLE stockout_runs_v2 AS 
            SELECT
            store_id,
            item_id,
            run_start_date,
            run_end_date,
            run_length,
            zero_rate_fraction_p,
            chance,
            stockout_flag,
            CASE WHEN run_length>56 THEN 'inactive' WHEN stockout_flag =1 AND run_length <=56 THEN 'stockout' ELSE 'normal' END AS run_type
            FROM stockout_runs""")
            
            

In [51]:
con.sql("""SELECT * FROM stockout_runs_v2 LIMIT 5""").df()

,store_id,item_id,run_start_date,run_end_date,run_length,zero_rate_fraction_p,chance,stockout_flag,run_type
0,WI_1,HOBBIES_1_187,2014-02-21,2014-02-27,7,0.840442,0.296180,0,normal
1,TX_1,HOUSEHOLD_2_316,2013-01-29,2013-02-04,7,0.591963,0.025472,0,normal
2,TX_1,FOODS_3_549,2015-06-14,2015-06-20,7,0.693413,0.077081,0,normal
3,WI_3,FOODS_3_104,2014-01-31,2014-02-06,7,0.573842,0.020490,0,normal
4,WI_2,FOODS_1_072,2013-11-20,2013-11-26,7,0.847178,0.313200,0,normal


** Add a column run_type to a new table stockout_runs_v2 (based on stockout_runs):

'inactive' when the run is longer than 56 days
'stockout' when the existing flag is 1 and the run is 56 days or shorter
'normal' otherwise

Then show, for each run_type: the number of runs, the total days and the average run length.

In [59]:
con.sql("""SELECT 
            run_type,
            COUNT(*) AS no_of_runs,
            SUM(run_length) AS total_days,
            AVG(run_length) AS avg_run_length,
            total_days*100.0/ SUM(total_days) OVER() AS pct_zero_days,
            no_of_runs*100.0/SUM(no_of_runs) OVER() AS pct_of_runs
            FROM stockout_runs_v2
            GROUP BY run_type
            ORDER BY run_type""").df()
            

,run_type,no_of_runs,total_days,avg_run_length,pct_zero_days,pct_of_runs
0,inactive,45437,6153451.0,135.428197,22.021908,0.687033
1,normal,6465068,19890614.0,3.076629,71.184328,97.755503
2,stockout,103003,1898341.0,18.429958,6.793764,1.557464


**Findings: stockout flag**
- 6.6M zero-sales runs found. 97.8% are short "normal" runs (avg 3 days), typical of intermittent demand.
- 103K runs (1.6%) are likely **stockouts**: too long to be chance for that item-store, avg 18 days, 1.9M zero days.
- 45K runs (0.7%) are **inactive** (longer than 56 days, avg 135 days) but hold 22% of all zero days. These look like items not stocked in that store despite having a price, a master-data issue.
- About 29% of zero on-sale days are not real zero demand. Forecasting and safety stock should exclude or correct them.

**Assumptions and limitations**
- 1-in-1,000 chance threshold and 56-day inactive boundary are business assumptions; real inventory or ranging data would be needed to validate them.
- The zero rate (p) includes the stockout and inactive runs themselves, which makes the rule less sensitive for items with frequent stockouts. A second pass could recompute p after removing flagged runs.
- 